<a href="https://colab.research.google.com/github/sameerkarur/Data_science/blob/main/06_IITK_AIML_Advanced_Generative_AI/demos/Lesson_06_LangChain_for_LLM_Application_Development_Part_2/Demo_03_LangChain_Memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **LangChain Memory**

### **Objective:**  
To explore different types of memory in LangChain and understand how each type retains, summarizes, or limits conversational context. This helps in building efficient AI systems that maintain relevant context while optimizing performance and memory usage.

---

### **Note:**  
- Before running any demo, ensure that the **requirements.txt** file is installed. This file contains all the required dependencies for **all demos and guided practices under Lesson_06**.
- If the dependencies were already installed earlier (after creating the virtual environment), there is no need to install them again. You can directly proceed with running the demo.
- Refer to Lesson_06 **Demo_01_Langchain_Loader_splitter_embeddings_vectorstore** Step 1 for creating a virtual environment and installing the requirements.txt 
- Ensure you select the right kernel **Python (myenv)** while running the demos
---


### **Steps to perform:**
1. Import the necessary modules  
2. Initialize the chat model  
3. Define ConversationBufferMemory  
4. Define ConversationBufferWindowMemory with a window size of 1  
5. Define ConversationTokenBufferMemory with a maximum token limit of 30  
6. Define ConversationSummaryBufferMemory with a maximum token limit of 100  

---


### **Step 1: Import the necessary modules**
- Import all required components for creating memory chains and managing conversational history


In [14]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables.history import RunnableWithMessageHistory
from pydantic import PrivateAttr
import textwrap
import warnings
warnings.filterwarnings("ignore")

- Define a helper function to print message history in a readable format

In [15]:
def print_clean_history(history):
    for m in history.messages:
        role = "User" if m.type == "human" else "AI"
        print(f"{role}: {m.content}")
    print()

### **Step 2: Initialize the chat model**
- Initialize the ChatOpenAI model with a deterministic configuration


In [16]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

In [17]:
base_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("placeholder", "{history}"),
    ("human", "{input}")
])

def build_chain(history):
    return RunnableWithMessageHistory(
        base_prompt | llm,
        lambda session_id: history,
        input_messages_key="input",
        history_messages_key="history",
    )

### **Step 3: Define ConversationBufferMemory**
- Purpose: Stores the entire conversation history without removing or summarizing any messages

In [18]:
print("### ConversationBufferMemory Output ###\n")

buffer_memory = ChatMessageHistory()
buffer_chain = build_chain(buffer_memory)

buffer_chain.invoke({"input": "Hello, my name is Alex"}, config={"configurable": {"session_id": "A"}})
buffer_chain.invoke({"input": "What is 2+2?"}, config={"configurable": {"session_id": "A"}})
buffer_chain.invoke({"input": "What is my name?"}, config={"configurable": {"session_id": "A"}})
buffer_chain.invoke({"input": "Add 10 to the above output"}, config={"configurable": {"session_id": "A"}})

print("Memory Buffer:")
print_clean_history(buffer_memory)

### ConversationBufferMemory Output ###

Memory Buffer:
User: Hello, my name is Alex
AI: Hello, Alex! How can I assist you today?
User: What is 2+2?
AI: 2 + 2 equals 4.
User: What is my name?
AI: Your name is Alex.
User: Add 10 to the above output
AI: Adding 10 to 4 gives you 14.



ConversationBufferMemory stores the entire conversation history without removing or summarizing anything.
Every message — both user and AI — is preserved and passed back into the model. This simulates ChatGPT’s typical behavior.

Output Explanation - The model remembers Alex’s name, it remembers the previous math result ("4"), and it uses the full conversation to respond correctly.

No information is lost.

- Full memory enables context-aware conversations.

- The model can reference earlier user messages.

- This is useful for assistants that need long-term conversation tracking.

### **Step 4: Define ConversationBufferWindowMemory with a window size of 1**



In [19]:
class WindowedHistory(ChatMessageHistory):
    _k: int = PrivateAttr()

    def __init__(self, k=1, **kwargs):
        super().__init__(**kwargs)
        self._k = k

    def add_message(self, message):
        super().add_message(message)
        # Keep only last k turns (each turn = human + AI)
        self.messages = self.messages[-2 * self._k:]

In [20]:
print("\n### ConversationBufferWindowMemory Output ###\n")

window_memory = WindowedHistory(k=1)
window_chain = build_chain(window_memory)

window_chain.invoke({"input": "Hello, my name is Alex"}, config={"configurable": {"session_id": "B"}})
window_chain.invoke({"input": "What is 2+2?"}, config={"configurable": {"session_id": "B"}})
window_chain.invoke({"input": "What is my name?"}, config={"configurable": {"session_id": "B"}})
window_chain.invoke({"input": "Add 10 to the above output"}, config={"configurable": {"session_id": "B"}})

print("Windowed Memory Buffer:")
print_clean_history(window_memory)


### ConversationBufferWindowMemory Output ###

Windowed Memory Buffer:
User: Add 10 to the above output
AI: Sure! If we add 10 to the previous output, it would be:

"I'm sorry, but I don't know your name. If you'd like to share it, feel free! 10" 

Let me know if you meant something else!



Explanation:
- Only the latest conversation turn is stored.
- Earlier context (like Alex’s name) is lost.
- Ideal for short or stateless conversations and low token usage scenarios.

### **Step 5: Define ConversationTokenBufferMemory with a maximum token limit of 30**




In [21]:
class TokenLimitedHistory(ChatMessageHistory):
    _max_tokens: int = PrivateAttr()

    def __init__(self, max_tokens=30, **kwargs):
        super().__init__(**kwargs)
        self._max_tokens = max_tokens

    def token_count(self):
        return sum(len(m.content.split()) for m in self.messages)

    def add_message(self, message):
        super().add_message(message)
        while self.token_count() > self._max_tokens:
            self.messages.pop(0)

In [22]:
print("\n### ConversationTokenBufferMemory Output ###\n")

token_memory = TokenLimitedHistory(max_tokens=30)
token_chain = build_chain(token_memory)

token_chain.invoke({"input": "Machine Learning is what?!"}, config={"configurable": {"session_id": "C"}})
token_chain.invoke({"input": "Neural Networks are what?"}, config={"configurable": {"session_id": "C"}})
token_chain.invoke({"input": "AI Assistants are what?"}, config={"configurable": {"session_id": "C"}})

print("Token-Limited Memory Buffer:")
print_clean_history(token_memory)


### ConversationTokenBufferMemory Output ###

Token-Limited Memory Buffer:



Explanation:
- The memory trims messages when total tokens exceed 30.
- Prevents context overflow, API overuse, and irrelevant long-term memory.
- Useful in production systems where token budgets must be tightly managed.

### **Step 6: Define ConversationSummaryBufferMemory with a maximum token limit of 100**





In [27]:
from pydantic import PrivateAttr
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage

class SummaryMemory(ChatMessageHistory):
    _llm: any = PrivateAttr()
    _max_turns: int = PrivateAttr()

    def __init__(self, llm, max_turns=4, **kwargs):
        super().__init__(**kwargs)
        self._llm = llm
        self._max_turns = max_turns

    def add_message(self, message):
        super().add_message(message)

        # When history grows beyond max turns, summarize it
        if len(self.messages) > self._max_turns:
            convo_text = "\n".join(
                f"{m.type.upper()}: {m.content}" 
                for m in self.messages
                if hasattr(m, "content")
            )

            summary = self._llm.invoke(
                f"Summarize this conversation:\n{convo_text}"
            ).content

            # Replace the history with the summary
            self.messages = [AIMessage(content=summary)]

In [28]:
print("\n### ConversationSummaryBufferMemory Output ###\n")

summary_memory = SummaryMemory(llm=llm, max_turns=4)
summary_chain = build_chain(summary_memory)

summary_chain.invoke({"input": "Hi"}, config={"configurable": {"session_id": "D"}})
summary_chain.invoke({"input": "What's up?"}, config={"configurable": {"session_id": "D"}})
summary_chain.invoke({"input": "What is on today's agenda?"}, config={"configurable": {"session_id": "D"}})
summary_chain.invoke({"input": "Tell me more."}, config={"configurable": {"session_id": "D"}})

print("Summary Memory Buffer:")
print_clean_history(summary_memory)


### ConversationSummaryBufferMemory Output ###

Summary Memory Buffer:
AI: The conversation begins with a greeting from the human, to which the AI responds warmly, offering assistance. The human then asks about the current agenda, indicating a desire to discuss specific topics.
AI: I don’t have access to real-time information or personal agendas, but I can help you create a plan or discuss topics you might want to cover today. Do you have specific tasks or subjects in mind?
User: Tell me more.
AI: Sure! Here are a few ways I can assist you:

1. **Planning and Organization**: If you have tasks or projects to manage, I can help you create a to-do list or outline a plan.

2. **Information and Research**: If you need information on a specific topic, I can provide summaries, explanations, or insights based on what I know.

3. **Problem-Solving**: If you're facing a challenge, I can help brainstorm solutions or provide advice.

4. **Learning and Education**: If you're interested in learning

Explanation:

- When history exceeds 4 messages, older context is summarized by the LLM.
- Maintains meaning without preserving every detail.
- Ideal for long sessions (For example:  customer support or personal assistants).

### **Conclusion**
By completing this demo, you learned how to use different types of memory mechanisms in LangChain

- ConversationBufferMemory: Retains the full history.
- ConversationBufferWindowMemory: Keeps only recent turns.
- ConversationTokenBufferMemory: Enforces a token-based cap.
- ConversationSummaryBufferMemory: Summarizes and compresses long interactions.

Each memory type offers unique trade-offs between context retention, performance, and token efficiency, enabling developers to design adaptive conversational systems.

---